# GridOps-SoD, suite v2 (main-track run)

Three wordings per attack, an adaptive injection (A1_adaptive) and three models, including a larger one.

**Before starting:** *Session options* -> **Accelerator = GPU T4 x2**, **Internet = On**.
Then **Save Version -> Save & Run All (Commit)** to run in the background (about 5 hours).
At the end, download `gridops_v2_results.zip` from the *Output* tab of the saved version.

In [ ]:
# 1. Code and dependencies
!git clone -q https://github.com/valeriobellandi/agentsod.git /kaggle/working/agentsod
%cd /kaggle/working/agentsod
!pip install -q pandapower
!apt-get -qq install -y zstd > /dev/null

In [ ]:
# 2. Ollama (OpenAI-compatible server on localhost:11434)
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)  # survives cell interrupts
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags"); break
    except Exception:
        time.sleep(2)
print("ollama up")

In [ ]:
# 3. Models: two 7-8B models of different families and a larger 14B model
MODELS = ["qwen2.5:7b", "llama3.1:8b", "qwen2.5:14b"]
for m in MODELS:
    !ollama pull {m}

In [ ]:
# 4. Pilot: 2 episodes per kind with the first model
!python -m experiments.gridops.run --suite v2 --model {MODELS[0]} --n 2 --workers 4 --out results/gridops/v2/pilot.jsonl
!python -m experiments.gridops.analyze results/gridops/v2/pilot.jsonl | cut -d, -f1-8

In [ ]:
# 5. Full run: 60 episodes per kind (6 kinds x 60 = 360 episodes, x 3 conditions) per model.
#    Resumable: if the session stops, run this cell again and it continues where it left off.
N = 60
for m in MODELS:
    out = f"results/gridops/v2/{m.replace(':', '-')}.jsonl"
    !python -m experiments.gridops.run --suite v2 --model {m} --n {N} --workers 4 --out {out}

In [ ]:
# 6. Summary, LaTeX table and per-variant breakdown
import glob
files = " ".join(sorted(f for f in glob.glob("results/gridops/v2/*.jsonl") if "pilot" not in f))
!python -m experiments.gridops.analyze {files} --csv results/gridops/v2/summary.csv --tex results/gridops/v2/table.tex | cut -d, -f1-8
!python -m experiments.gridops.analyze {files} --by-variant --csv results/gridops/v2/summary_by_variant.csv > /dev/null

In [ ]:
# 7. Package the results for download
!cp /kaggle/working/ollama.log /kaggle/working/agentsod/results/gridops/v2/ 2>/dev/null
!cd /kaggle/working/agentsod && zip -qr /kaggle/working/gridops_v2_results.zip results/gridops/v2 -x '*ollama.log'
!ls -la /kaggle/working/gridops_v2_results.zip